# CFX — Calibrated & Faithful eXplanation

The executable driver for the explanation layer. Every module it calls is a `.py`
file under `src/fsvnet/`; this notebook only orchestrates and prints tables.

**Read this before running anything.** The honest numbers for this detector are
in `docs/FINDINGS.md`: identity-disjoint channel probes sit at 0.51–0.59 AUC, and
`results/ablations.csv` records the full five-branch model at **−0.111 AUC
against its own no-TSC/no-LMF ablation**, resolved at p≈0.

That is what this layer is built around. It does not try to explain why a video
is fake. It decides whether the model has earned the right to explain itself, and
refuses when it has not. **Most clips should come out as abstentions.** If they
do not, `assert_coverage_sane` fails the run — because a confident-looking report
from a leaky split is the most expensive failure mode in this project.

Order of operations:

1. verify the modules against the stub model (no checkpoints needed)
2. `cfx_fit.py` — calibration, abstention band, reliability matrix, arbitration weights (**val only**)
3. `cfx_eval.py` — faithfulness benchmark, narration quality, the correctness test (**test**)
4. inspect one card in both registers
5. the report tables

## 0. Setup

In [ ]:
import sys, json, subprocess
from pathlib import Path
import numpy as np, pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
OUT = ROOT / "results" / "cfx"
print("root:", ROOT)

import torch
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"gpu: {p.name} {p.total_memory/1e9:.1f} GB")
else:
    print("no CUDA -- pass --cpu to the scripts below; they use the documented "
          "CPU profile (~400 forwards/clip instead of ~2,800)")

## 1. Verify the layer before trusting it

142 checks against a stub model whose answers are known in advance: exact Shapley
must find the planted decisive branch and give the noise branch zero credit with
an efficiency residual of ~0; integrated gradients must localise the planted
frame *and* region; a real attribution must beat a random one on deletion and a
random one must not; and a deliberately hostile generator that cites no evidence
and writes "clearly a deepfake" must not reach the user.

Run this first. If it fails, nothing below means anything.

In [ ]:
r = subprocess.run([sys.executable, str(ROOT / "scripts" / "test_cfx.py")],
                   capture_output=True, text=True)
print(r.stdout[-3000:])
if r.returncode != 0:
    print(r.stderr[-2000:])
    raise SystemExit("CFX self-test failed -- fix before going further")

## 2. Fit on val

Calibrators, the abstention band, the reliability matrix and the arbitration
weights. **Nothing here touches test.**

Start with `--limit 32` to confirm the checkpoints load and the shapes line up;
the full pass scores four checkpoints over val and test plus one ablation pass
per channel, so it is a coffee-length job on the 4060.

In [ ]:
SMOKE = True   # flip to False for the real run

cmd = [sys.executable, str(ROOT / "scripts" / "cfx_fit.py"), "--alpha", "0.10"]
if SMOKE:
    cmd += ["--limit", "32"]
r = subprocess.run(cmd, capture_output=True, text=True)
print(r.stdout[-6000:])
print(r.stderr[-1500:] if r.returncode else "")

### What to look at in that output

**Calibration.** Compare equal-mass ECE against equal-width. At an 86/14 prior the
equal-width figure flatters the model, so the equal-mass one is the headline.

**Resolution.** In the Brier decomposition, resolution is the only term that
measures whether the model knows anything, and calibration cannot improve it. A
model with good ECE and resolution ≈ 0 is honestly predicting the base rate for
everything. That is a finding, not a bug — but it must be reported as one.

**The abstention band.** If it abstains on nearly everything, that is the weak
detector being represented correctly. Raise `--alpha` if you need coverage, and
say in the report that you did.

**Selective accuracy vs 0.864.** "Always say fake" scores 0.864 accuracy on this
data. Any selective accuracy below that line is not an achievement.

In [ ]:
cal = OUT / "cfx_calibration.csv"
if cal.exists():
    df = pd.read_csv(cal)
    display(df[["condition", "stage", "AUC", "ECE (equal-mass)",
                "ECE (equal-width)", "Brier", "reliability", "resolution",
                "calibrator"]].round(4))
    flat = df[(df.stage == "calibrated") & (df.resolution < 0.01)]
    if len(flat):
        print("\nRESOLUTION NEAR ZERO for:", ", ".join(flat.condition))
        print("These conditions are well-calibrated and carry almost no "
              "discriminative information. Report both facts together.")
else:
    print("run the cell above first")

In [ ]:
sel = OUT / "cfx_selective.csv"
if sel.exists():
    s = pd.read_csv(sel)
    display(s[["condition", "coverage", "accuracy", "risk",
               "majority_class_baseline", "beats_majority", "AURC",
               "E-AURC"]].round(4))
    if not s.beats_majority.any():
        print("\nNo condition beats the majority-class baseline at any coverage.")
        print("That is the result. It says the confidence ranking carries no "
              "usable information on this data, and it belongs in the report "
              "next to the risk-coverage curve.")
else:
    print("run the fit cell first")

### The risk–coverage curve

The headline figure of this whole layer. The 0.864 line is drawn on it because
without that line the selective accuracy numbers read far better than they are.

In [ ]:
from fsvnet.selective import risk_coverage_curve, majority_class_baseline
import matplotlib.pyplot as plt
from fsvnet import viz
viz.use_style()

fig, ax = plt.subplots(figsize=(7, 4.2))
drew = False
for f in sorted(OUT.glob("scores_*.npz")):
    cid = f.stem.split("_")[-1]
    d = np.load(f)
    if "y_test" not in d:
        continue
    rc = risk_coverage_curve(d["y_test"], d["p_test"])
    ax.plot(rc["coverage"], rc["accuracy"], label=cid, lw=1.8)
    base = majority_class_baseline(d["y_test"])
    drew = True
if drew:
    ax.axhline(base, ls="--", lw=1.2, color=viz.INK_MUTED,
               label=f'"always fake" = {base:.3f}')
    ax.set_xlabel("coverage (fraction of clips answered)")
    ax.set_ylabel("accuracy on the answered clips")
    ax.set_title("Selective accuracy vs coverage, identity-disjoint test split")
    ax.legend(fontsize=8)
    viz.despine(ax)
    fig.tight_layout()
    fig.savefig(ROOT / "results" / "visualizations" / "cfx_risk_coverage.png", dpi=150)
else:
    print("no test scores yet")

### The reliability matrix

Which model's version of which evidence channel deserves to be believed.

Note what `cfx_fit.py` leaves blank: faithfulness and stability are per-clip
measurements and come from `cfx_eval.py`. Until that has run, every channel scores
zero and CFX abstains on everything — the correct default, because no channel has
earned a voice yet.

**A prediction written down in advance:** given the ablation table, arbitration
should route most weight to **B0** and score TSC and LMF low. If it does, that is
the dataset-level ablation finding rediscovered automatically, per clip. Check it;
don't narrate it afterwards.

In [ ]:
from fsvnet.arbitrate import ReliabilityMatrix
f = OUT / "evidence_reliability.json"
if f.exists():
    m = ReliabilityMatrix.load(f)
    print(m.summary())
    print()
    for n in m.notes:
        print("note:", n)
else:
    print("run the fit cell first")

## 3. Measure on test

The faithfulness benchmark, the narration checks, and the question the whole
design rests on: does explanation faithfulness predict correctness?

`--n` clips are drawn stratified by label with a fixed seed. The full pass is
~2,800 forwards per clip, so this is a subsample and the CSVs carry `n` and the
seed — the report says "on a stratified 200-clip subsample", never implying full
coverage.

Add `--narrate` to load Qwen and measure the layman register. Without it, both
registers come from the deterministic template and `template_fallback_rate` is
100% by construction rather than by failure.

In [ ]:
N = 8            # raise to 200 for the reportable run
NARRATE = False  # True loads Qwen2.5-VL-3B (NF4, ~2.2 GB)

cmd = [sys.executable, str(ROOT / "scripts" / "cfx_eval.py"), "--n", str(N),
       "--no-strict-coverage"]
if NARRATE:
    cmd.append("--narrate")
if not torch.cuda.is_available():
    cmd.append("--cpu")
r = subprocess.run(cmd, capture_output=True, text=True)
print(r.stdout[-8000:])
print(r.stderr[-1500:] if r.returncode else "")

### Faithfulness, per method

`beats_random_rate` is the gate, and it is the column to read first. A method
below 0.5 here is not an explanation, and the honest thing to do is report it as
a row in this table rather than quietly not mentioning it. Several methods failing
this test is a normal outcome and a publishable one.

In [ ]:
f = OUT / "cfx_faithfulness.csv"
if f.exists():
    fd = pd.read_csv(f)
    display(fd[["method", "modality", "n", "beats_random_rate", "mean_margin",
                "admissible_rate", "mean_solidity", "verdict"]].round(4))
    losers = fd[fd.beats_random_rate <= 0.5]
    if len(losers):
        print("\nMethods that do NOT beat a random ordering of the same units:")
        for _, r_ in losers.iterrows():
            print(f"  - {r_.method} ({r_.beats_random_rate:.0%} of clips)")
        print("Report these. An explanation indistinguishable from noise is a "
              "result about the method, not a gap in the table.")
else:
    print("run the eval cell first")

### Where the clips landed

The four-quadrant policy, counted. At this detector's honest accuracy the
low-confidence row should dominate. If coverage is above 90%, the run wrote a
`COVERAGE_WARNING.txt` and the numbers should not be reported until the split has
been re-checked for identity leakage.

In [ ]:
q = OUT / "cfx_quadrants.csv"
if q.exists():
    display(pd.read_csv(q))
w = OUT / "COVERAGE_WARNING.txt"
if w.exists():
    print("\n!!", w.read_text())

### Narration quality

Four numbers that turn "written for non-technical readers" into a measurement:

| column | what it means |
|---|---|
| `groundedness` | share of generated sentences that survived the claim-link filter |
| `numeral_ok` | every number in the technical register appears in the card |
| `back_translation_ok` | the layman text still refers to the evidence it cited |
| `fk_grade` | Flesch–Kincaid reading level (target ≤ 8, ceiling 10) |

`n_dropped` is worth reading directly: those are sentences a model wrote that
nothing measured supported, and they were deleted rather than rewritten.

In [ ]:
f = OUT / "cfx_narration.csv"
if f.exists():
    nd = pd.read_csv(f)
    display(pd.DataFrame([{
        "clips": len(nd),
        "groundedness": nd.groundedness.mean(),
        "numeral fidelity": nd.numeral_ok.mean(),
        "back-translation pass": nd.back_translation_ok.mean(),
        "template fallback": nd.fell_back_to_template.mean(),
        "mean FK grade": nd.fk_grade.mean(),
        "sentences dropped": int(nd.n_dropped.sum()),
    }]).round(3))
    bad = nd[~nd.numeral_ok]
    if len(bad):
        print("\nNumeral-fidelity failures (a template bug, not a style issue):")
        display(bad[["clip_id", "unsupported_numerals"]])
else:
    print("run the eval cell first")

### The testable claim

Is the aggregate faithfulness score higher on clips the model got right than on
clips it got wrong?

* **resolved and positive** → faithfulness is itself a confidence signal and
  belongs in the selective score. A novel, measured result.
* **not resolved** → report the null. It is the first thing anyone would ask about
  this design, and the answer is a finding either way.

In [ ]:
f = OUT / "cfx_correctness.json"
if f.exists():
    r_ = json.loads(f.read_text())
    print(json.dumps(r_, indent=2))
    if r_.get("resolved"):
        print("\n-> faithfulness carries confidence information. Add it to the "
              "selective score and re-run the risk-coverage curve.")
    else:
        print("\n-> the null. Faithfulness does not predict correctness on this "
              "evidence. Report it with the CI; do not re-run with a different "
              "n until it comes out positive.")

## 4. One card, both registers

The toggle in the Streamlit app is exactly this: two strings that were both
written when the clip was analysed. Nothing is regenerated on switching, which is
why it is instant and why the two registers can never disagree about a fact.

In [ ]:
cards = sorted((OUT / "cards").glob("*.json"))
print(f"{len(cards)} cards")
if cards:
    c = json.loads(cards[0].read_text())
    v, nar = c["verdict"], (c.get("narration") or {})
    print(f"\nclip      : {c['clip_id']}")
    print(f"decision  : {v['decision']}   p={v['p_fake_calibrated']:.3f}   "
          f"band=[{v['band'][0]:.3f}, {v['band'][1]:.3f}]")
    print(f"quadrant  : {c['quadrant']}")
    print(f"\nLAYMAN\n  {nar.get('layman', '')}")
    print(f"\nTECHNICAL\n  {nar.get('technical', '')}")
    print("\n--- hidden from the user, one click away in the app ---")
    print(f"answering model : {v['answering_model']}")
    print(f"routing reason  : {v['routing_reason']}")
    sh = c.get("shapley") or {}
    print(f"shapley         : {sh.get('values')}")
    print(f"best explanation: {(c.get('best') or {}).get('method')} "
          f"(grade {(c.get('best') or {}).get('grade')})")

## 5. The front end

    streamlit run streamlit/cfx_app.py

Default view: verdict, confidence with its band, one explanation, and the
technical ⇄ plain-language toggle. Everything else — routing, solidity factor by
factor, Shapley values and interactions, every candidate with its margin over
random, the raw card — sits collapsed under **Audit**. Collapsed rather than
removed, because the examiner *is* the technical audience, and "the interesting
parts are one click away" demos better than "trust me".

Two things the UI will not do: expose model selection as a setting (arbitration is
the contribution — a dropdown gives it away), and show a confidence number without
its band.

## 6. What goes in the chapter

1. Calibration table — pre/post, equal-mass **and** equal-width ECE, MCE, Brier decomposition; val and test.
2. Risk–coverage curve and AURC, selective accuracy at 20/40/60/80% coverage, with the 0.864 majority line on the same axes.
3. The reliability matrix (models × channels).
4. Arbitration vs B0, paired bootstrap, on test — **including if it loses**, in which case ship B0 and say so.
5. Faithfulness table with every method's margin over random. Expect several not to beat it.
6. Narration quality: groundedness, numeral fidelity, FK grade, fallback rate.
7. The faithfulness-vs-correctness test.
8. Quadrant counts — how often the system declines to explain, which is the design working.

One cheap addition worth an afternoon: show the plain-language explanation to
10–15 non-technical people, show verdict-only to another group, and ask both how
often they think the system is right. If the explanation inflates their estimate,
the layman register is doing harm and you have measured it. Almost nobody in the
deepfake-XAI literature runs that check.